# Model verification Check 

This is a test that the saved pipeline model matched the original model. 

Expected metrics:
- R^2 = 0.9271
- RMSE = 1.2117


In [3]:
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import root_mean_squared_error, r2_score
from pathlib import Path

In [5]:
 # use Path.cwd() to get the current working directory.
current_dir = Path.cwd(__file__).resolve().parent if "__file__" in locals() else Path.cwd()

# Navigate up two levels from notebook/model/ to the root, then down into data/final/
model_path = current_dir.parents[1] / "data" / "final" / "best_model_pipeline.joblib"

print(f"File exists: {model_path.exists()}")

File exists: True


In [8]:
# import X_test and y_test from data/final/
X_test_path = current_dir.parents[1] / "data" / "final" /"X_test.csv"
y_test_path = current_dir.parents[1] / "data" / "final"/"y_test.csv"

X_test = pd.read_csv(X_test_path)
y_test = pd.read_csv(y_test_path)

In [9]:
def verify_serialized_pipeline(model_path, X_test, y_test, expected_rmse):
    """
    Loads the serialized pipeline and confirms it reproduces 
    the exact test performance reported in the paper.
    """
    # 1. Load the serialized pipeline in its entirety
    loaded_pipeline = joblib.load(model_path)
    
    # 2. Isolate features (dropping ID columns if present)
    ID_COLS = ["fipscode", "locationname", "stateabbr", "State", "County"]
    X_te = X_test.drop(columns=[c for c in ID_COLS if c in X_test.columns], errors="ignore")
    y_te = y_test.squeeze()
    
    # 3. Generate predictions using the reloaded artifact
    y_pred_reloaded = loaded_pipeline.predict(X_te)
    
    # 4. Calculate metrics
    reloaded_r2 = r2_score(y_te, y_pred_reloaded)
    reloaded_rmse = root_mean_squared_error(y_te, y_pred_reloaded)
    
    # 5. Assert / Confirm exact match
    print(f"Loaded Pipeline Test R²: {reloaded_r2:.6f}")
    print(f"Loaded Pipeline Test RMSE: {reloaded_rmse:.6f}")
    
    # Check if RMSE matches your paper's reported metric (e.g., within standard floating-point tolerance)
    assert np.isclose(reloaded_rmse, expected_rmse, atol=1e-5), "Mismatch detected between reloaded pipeline and paper evaluation!"
    print("✅ Verification Passed: Serialized pipeline reproduces expected test metrics exactly.")

verify_serialized_pipeline(model_path, X_test, y_test, expected_rmse=1.2117)

Loaded Pipeline Test R²: 0.927059
Loaded Pipeline Test RMSE: 1.211715
✅ Verification Passed: Serialized pipeline reproduces expected test metrics exactly.
